# 13 · Task model with weather

The winner's time and the share of pilots in goal from what a task committee knows before the task (as in notebook 10) **plus the weather along the route during the task**: boundary layer and usable height, cloud base, CAPE, heat flux, sunshine, cloud, dry air, lapse rate, wind at 100 m and 700 hPa and its tailwind and crosswind along each leg, ground temperature and soil moisture, rain the day before (ERA5, the stand-in for the forecast).

Both models, with and without weather, are scored on the same folds (competitions left out, sites left out), so the difference is what the weather adds. It also writes, for the web page, the model as JSON (it runs in the browser) and each site's tasks with their weather, the predictions made without seeing that competition, and what happened.

Writes `database/models/task_v1/` and `database/web_data/task_model.json`, `database/web_data/tasks/`. About 10–15 minutes; High-RAM recommended (the weather tables are large).

## 1. Settings

In [ ]:
DRIVE_FOLDER_ID = "1_JO0XMMQ4NKVfpeA8uKQRkZZPNC4sqZ7"   # project folder in Google Drive (the id at the end of its link)
REPO_URL = "https://github.com/RobertojonesFAI/Pre-task-analyzer.git"
REPO = "/content/Pre-task-analyzer"

## 2. Mount Drive, get the code

In [ ]:
import pandas as pd
import os, shutil, subprocess
from google.colab import auth, drive
from googleapiclient.discovery import build

auth.authenticate_user()
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
service = build("drive", "v3")
FIELDS = "id, name, parents, driveId, ownedByMe"


def path_up(file_id, root):
    """Folder names from a file up to the top of its drive, joined under root."""
    names, current = [], file_id
    while True:
        meta = service.files().get(fileId=current, fields=FIELDS, supportsAllDrives=True).execute()
        if not meta.get("parents"):
            return os.path.join(root, *reversed(names))
        names.append(meta["name"])
        current = meta["parents"][0]


def drive_path(folder_id):
    """A writable local path for a Drive folder, from its id."""
    meta = service.files().get(fileId=folder_id, fields=FIELDS, supportsAllDrives=True).execute()
    if meta.get("driveId"):  # in a shared drive
        drive_name = service.drives().get(driveId=meta["driveId"]).execute()["name"]
        return path_up(folder_id, f"/content/drive/Shareddrives/{drive_name}")
    if meta.get("ownedByMe"):  # in my own Drive
        return path_up(folder_id, "/content/drive/MyDrive")
    # Shared with me: write through a shortcut to it in my Drive
    page = None
    while True:
        res = service.files().list(
            q="mimeType='application/vnd.google-apps.shortcut' and trashed=false",
            fields="nextPageToken, files(id, shortcutDetails)", pageToken=page).execute()
        for f in res.get("files", []):
            if f.get("shortcutDetails", {}).get("targetId") == folder_id:
                return path_up(f["id"], "/content/drive/MyDrive")
        page = res.get("nextPageToken")
        if not page:
            raise SystemExit(f"'{meta['name']}' is shared with you. Add a shortcut to it in your Drive "
                             "(right click -> Organize -> Add shortcut), then rerun this cell.")


ROOT = drive_path(DRIVE_FOLDER_ID)
DB = os.path.join(ROOT, "database")
assert os.path.isfile(os.path.join(DB, "task_fields.parquet")), "Run notebook 10 first: database/task_fields.parquet is missing"
subprocess.run(["pip", "install", "-q", "lightgbm"], check=True)
print("Database in", DB)

if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)


def run(*cmd):
    """Run a script and show its output; stopping the cell also stops the script, and a script
    that fails stops Run All."""
    proc = subprocess.Popen(["python3", "-u", *cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in proc.stdout:
            print(line, end="")
        code = proc.wait()
    except KeyboardInterrupt:
        proc.terminate()
        proc.wait()
        print("Stopped.")
        raise
    if code != 0:
        raise RuntimeError(f"{os.path.basename(cmd[0])} stopped with an error (exit code {code}); see the lines above")
    return code

## 3. Train and score

Time error: mean minutes between the predicted and the real winner's time; within 15 / 30 min: share of tasks that close. Goal share error in points (percentage points).

In [ ]:
run(f"{REPO}/scripts/task_model.py", DB)

## 4. Charts

In [ ]:
import matplotlib.pyplot as plt
p = pd.read_parquet(os.path.join(DB, "models", "task_v1", "predictions.parquet")).dropna(subset=["winner_h", "pred_winner_kmh", "base_winner_kmh"])
fig, ax = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)
for a, col, title in zip(ax, ["base_winner_kmh", "pred_winner_kmh"], ["without weather", "with weather"]):
    pred_h = p["ss_km"] / p[col]
    err = (pred_h - p["winner_h"]).abs().mean() * 60
    a.scatter(p["winner_h"], pred_h, s=8, color="#2a78d6", alpha=0.5)
    lim = [0, max(p["winner_h"].max(), pred_h.max())]
    a.plot(lim, lim, color="#c3c2b7", lw=1)
    a.set(title=f"{title}: {err:.0f} min mean error", xlabel="winner's time (h)")
    a.spines[["top", "right"]].set_visible(False)
ax[0].set_ylabel("predicted, competitions left out (h)")
plt.tight_layout(); plt.show()